In [ ]:
# ---- Cell 1: Install packages (run once per project, then comment out) ----
# Uncomment and run this cell ONCE on a fresh venv, then comment it out again.
# These installs can take 3-10 minutes on first run — sentence-transformers downloads model weights.

# !pip install pandas numpy matplotlib seaborn scikit-learn pyreadr
# !pip install sentence-transformers          # Core library — encodes text into embedding vectors
# !pip install torch                           # PyTorch backend — sentence-transformers runs on top of it
# !pip install faiss-cpu                       # Facebook AI Similarity Search — fast vector lookup (CPU version)
#                                              # If you have a GPU and want GPU acceleration:
# !pip install faiss-gpu                       # Use this instead of faiss-cpu on your RTX 4060

# ---- Note on first-run model download ----
# The first time Cell 4 runs, it will download the 'all-MiniLM-L6-v2' model (~90MB) from Hugging Face.
# This requires internet and takes 1-3 minutes. After that, the model is cached locally and
# loads instantly. The cache lives at: ~/.cache/huggingface/hub/
print("Cell 1: Package installs (commented out). Run this cell once if starting fresh.")

In [ ]:
# ============================================================
# 17 — Embeddings and Semantic Search
# ============================================================
# Purpose:  Teach what an embedding is — a vector of numbers that
#           encodes MEANING, not just identity. Build a semantic
#           search engine over synthetic clinical notes. Introduce
#           the concept of RAG (Retrieval-Augmented Generation) as
#           the bridge from search to AI assistants.
# Dataset:  Data/ml_teaching_data.csv  (our 25,000-patient dataset)
#           Synthetic clinical notes generated in this script from
#           structured patient records (same approach as Script 16).
# Kernel:   .venv (Python 3.10+)
# GPU:      RTX 4060 — sentence-transformers will use CUDA automatically
# ============================================================

# ---- What This Script Teaches ----
# 1. What an embedding is: a point in high-dimensional space that captures meaning
# 2. Why 'myocardial infarction' and 'heart attack' end up NEAR each other in that space
# 3. Cosine similarity: how to measure 'closeness' between two meaning-vectors
# 4. How to encode thousands of texts and search them by semantic meaning (not keywords)
# 5. FAISS: the index structure that makes vector search fast at scale
# 6. RAG (Retrieval-Augmented Generation): how LLMs use semantic search to find context

# ---- Why Python (not R)? ----
# The sentence-transformers library — and the pre-trained BERT/SBERT models it provides —
# live entirely in the Python ecosystem. The R equivalent (text package) is a thin wrapper
# that still calls Python under the hood. We use Python directly for full control,
# better error messages, and access to FAISS for fast vector search.

import pandas as pd          # DataFrames — like R's tibble
import numpy as np           # Numerical arrays and linear algebra
import matplotlib.pyplot as plt  # Base plotting — like R's base graphics
import matplotlib.patches as mpatches
import seaborn as sns        # Statistical plots — like R's ggplot2
from sklearn.decomposition import PCA      # Dimensionality reduction for visualization
from sklearn.metrics.pairwise import cosine_similarity  # Measure vector closeness
import os
import random
import warnings
warnings.filterwarnings('ignore')

# ---- GPU detection ----
# sentence-transformers uses PyTorch, which detects CUDA automatically.
# On your RTX 4060 this will print 'cuda' and embeddings will encode ~10x faster.
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# ---- Constants ----
SEED           = 42
N_NOTES        = 2000        # Number of synthetic clinical notes to generate and index
N_RESULTS      = 5           # How many similar notes to return for each query
EMBED_MODEL    = "all-MiniLM-L6-v2"  # Fast, high-quality SBERT model (~90MB)
                             # Produces 384-dimensional embeddings
                             # Trained on 1B+ sentence pairs to understand semantic similarity

# ---- Color palette (matches R scripts) ----
BLUE  = "#2196F3"   # Negative / no disease / low risk
RED   = "#F44336"   # Positive / disease / high risk
GREEN = "#4CAF50"   # Correct / match / success
GREY  = "#9E9E9E"   # Neutral / background

# ---- Output directory ----
OUT_DIR = "Outputs/Embeddings_Semantic_Search"
os.makedirs(OUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

print(f"Output directory: {OUT_DIR}")
print(f"Will generate {N_NOTES} synthetic clinical notes and build a semantic search index.")

In [ ]:
# ============================================================
# CELL 3: DATA LOADING + SYNTHETIC NOTE GENERATION
# ============================================================
# We load the structured patient dataset and convert each row into
# a free-text clinical note — like a doctor's chart entry.
#
# WHY generate notes from structured data?
# Real clinical notes contain the same information as our structured
# variables, but expressed in natural language with variation:
# 'the patient presents with elevated blood pressure' vs
# 'BP elevated at 148/90' vs 'hypertensive at presentation'
# Embeddings handle this variation; keyword search does not.

# ---- Load structured dataset ----
print("Loading dataset...")
try:
    import pyreadr
    ml_data = pyreadr.read_r("Data/ml_teaching_data.rds")[None]
    print("Loaded from .rds via pyreadr")
except Exception:
    ml_data = pd.read_csv("Data/ml_teaching_data.csv")
    print("Loaded from .csv")

print(f"Dataset shape: {ml_data.shape}")

# ---- Sample N_NOTES patients ----
# We use a random sample to keep encoding time reasonable.
# The full 25,000 would still work — it would just take ~30s to encode.
sample_df = ml_data.sample(n=N_NOTES, random_state=SEED).reset_index(drop=True)
print(f"\nSampled {N_NOTES} patients for note generation.")

# ---- Clinical note template engine ----
# We use vocabulary variation to make notes feel realistic:
# - Multiple ways to say the same thing (synonyms)
# - Occasional clinical abbreviations (HbA1c, CAD, SOB)
# - Randomly selected sentence structures
# This is what makes SEMANTIC search valuable — keyword search
# would miss 'myocardial infarction' when you search 'heart attack'.

def generate_clinical_note(row: pd.Series, patient_id: int) -> str:
    """Convert a structured patient row into a free-text clinical note.
    
    The note varies vocabulary so semantic search is genuinely needed.
    Keyword search on these notes would miss synonyms; embeddings won't.
    """
    r = row
    
    # ---- Age + sex header ----
    sex_word = random.choice(["male", "man"]) if str(r.get('sex','Male')) == 'Male' else random.choice(["female", "woman"])
    age_phrase = random.choice([
        f"Patient is a {int(r['age'])}-year-old {sex_word}",
        f"A {int(r['age'])}-year-old {sex_word} presents",
        f"{int(r['age'])}yo {sex_word}"
    ])

    # ---- Smoking ----
    smoking_val = str(r.get('smoking_status', 'Never')).strip()
    smoking_phrases = {
        'Current': random.choice(["active smoker", "current cigarette smoker", "tobacco use ongoing"]),
        'Former':  random.choice(["ex-smoker", "former tobacco user", "quit smoking previously"]),
        'Never':   random.choice(["non-smoker", "never smoked", "no tobacco history"])
    }
    smoking = smoking_phrases.get(smoking_val, "non-smoker")

    # ---- BMI ----
    bmi = float(r['bmi'])
    if bmi >= 35:
        bmi_desc = random.choice(["severely obese", "class II obesity", "BMI critically elevated"])
    elif bmi >= 30:
        bmi_desc = random.choice(["obese", "obesity noted", "BMI in obese range"])
    elif bmi >= 25:
        bmi_desc = random.choice(["overweight", "BMI above normal", "mild weight excess"])
    else:
        bmi_desc = random.choice(["normal weight", "BMI within range", "healthy weight"])

    # ---- Blood pressure ----
    sbp = float(r['sbp'])
    if sbp >= 140:
        bp_desc = random.choice([
            f"hypertensive (BP {int(sbp)}/{int(r['dbp'])})",
            f"elevated blood pressure at {int(sbp)}/{int(r['dbp'])}",
            f"BP {int(sbp)}/{int(r['dbp'])} — hypertension"
        ])
    elif sbp >= 120:
        bp_desc = f"borderline BP ({int(sbp)}/{int(r['dbp'])})"
    else:
        bp_desc = f"normal BP ({int(sbp)}/{int(r['dbp'])})"

    # ---- Glucose / diabetes ----
    glucose = float(r['fasting_glucose'])
    has_diabetes = int(r.get('diabetes', 0)) == 1
    if has_diabetes:
        glucose_desc = random.choice([
            "known diabetic, glucose management ongoing",
            "diabetes mellitus, fasting glucose elevated",
            "T2DM diagnosed, on glucose control"
        ])
    elif glucose >= 100:
        glucose_desc = random.choice([
            f"pre-diabetic range fasting glucose ({int(glucose)} mg/dL)",
            f"impaired fasting glucose at {int(glucose)}",
            f"borderline glucose {int(glucose)} mg/dL"
        ])
    else:
        glucose_desc = f"fasting glucose normal ({int(glucose)} mg/dL)"

    # ---- Heart disease ----
    has_hd = int(r.get('heart_disease', 0)) == 1
    if has_hd:
        hd_desc = random.choice([
            "History of coronary artery disease.",
            "Known myocardial infarction in chart.",
            "Cardiac history: prior heart attack.",
            "CAD on problem list.",
            "Documented ischemic heart disease."
        ])
    else:
        hd_desc = ""

    # ---- Cholesterol ----
    chol = float(r['cholesterol'])
    if chol >= 240:
        chol_desc = random.choice([
            f"high cholesterol ({int(chol)} mg/dL)",
            f"hypercholesterolemia, total chol {int(chol)}",
            f"elevated lipids ({int(chol)} mg/dL)"
        ])
    elif chol >= 200:
        chol_desc = f"borderline cholesterol ({int(chol)} mg/dL)"
    else:
        chol_desc = f"cholesterol within normal limits ({int(chol)} mg/dL)"

    # ---- Medications ----
    meds = int(r.get('medication_count', 0))
    if meds >= 5:
        med_desc = f"polypharmacy ({meds} medications)"
    elif meds > 0:
        med_desc = f"{meds} medication(s) on formulary"
    else:
        med_desc = "no current medications"

    # ---- Assemble note ----
    note = (
        f"{age_phrase} with {smoking}, {bmi_desc}. "
        f"{bp_desc.capitalize()}, {chol_desc}, {glucose_desc}. "
        f"{hd_desc} "
        f"Currently on {med_desc}. "
        f"Risk profile reviewed at visit ID {patient_id}."
    ).strip()

    return note


# ---- Generate notes ----
print("\nGenerating synthetic clinical notes...")
notes = [
    generate_clinical_note(sample_df.iloc[i], i)
    for i in range(len(sample_df))
]
sample_df['clinical_note'] = notes

# ---- Show examples ----
print(f"\nGenerated {len(notes)} clinical notes.")
print("\n--- Example notes (first 3) ---")
for i, note in enumerate(notes[:3]):
    print(f"\nNote {i+1} (heart_disease={sample_df.iloc[i]['heart_disease']}):\n  {note}")

# ---- Show vocabulary variation: same concept, different words ----
# This is exactly why we need embeddings instead of keyword search:
heart_notes = sample_df[sample_df['heart_disease']==1]['clinical_note'].head(5)
print("\n--- Heart disease notes (notice vocabulary variation across rows) ---")
for note in heart_notes:
    # Extract just the cardiac sentence to highlight variation
    for phrase in ['coronary', 'myocardial', 'heart attack', 'CAD', 'ischemic']:
        if phrase.lower() in note.lower():
            print(f"  Found '{phrase}': {note[:120]}...")
            break

print("\nNotice: all above notes mean 'has heart disease' but use different words.")
print("Keyword search for 'heart attack' would MISS 'myocardial infarction' notes.")
print("Embedding-based search finds ALL of them.")

In [ ]:
# ============================================================
# CELL 4: WHAT IS AN EMBEDDING? + ENCODE + BUILD VECTOR INDEX
# ============================================================
#
# ---- What is an Embedding? ----
# Imagine a giant room where every word or sentence has a LOCATION.
# Sentences that MEAN similar things are placed NEAR each other.
# 'The patient has a heart attack'           → position [0.12, -0.45, 0.78, ...]
# 'Myocardial infarction documented'         → position [0.11, -0.43, 0.77, ...]
# 'Patient enjoys hiking on weekends'        → position [-0.82, 0.31, -0.55, ...]
#
# The two cardiac sentences have SIMILAR positions; the hiking sentence is FAR away.
# This is because the model was trained on massive text and learned that
# 'heart attack' and 'myocardial infarction' appear in the same contexts.
#
# Each position is a VECTOR — a list of numbers (384 numbers for our model).
# That 384-dimensional space is the 'embedding space'.
#
# ---- What is Cosine Similarity? ----
# To compare two vectors (two positions in embedding space), we measure
# the ANGLE between them, not the distance. Why angle?
# - Angle ignores length (a short summary and a long note about the same
#   topic should still be 'close', even though one is much longer)
# - cos(0°) = 1.0  → identical direction (same meaning)
# - cos(90°) = 0.0 → perpendicular (unrelated)
# - cos(180°) = -1.0 → opposite direction (opposite meaning)
# In practice, semantic similarity scores are between 0 and 1.
#
# ---- What is FAISS? ----
# If we have 2,000 notes and a query, brute-force comparison means
# computing cosine similarity with ALL 2,000 notes.
# FAISS (Facebook AI Similarity Search) builds an INDEX of the vectors
# that allows approximate nearest-neighbour lookup in milliseconds,
# even with millions of vectors. Think of it like a spatial hashtable.

from sentence_transformers import SentenceTransformer
import faiss  # Fast vector similarity search — required for production-scale RAG systems

# ---- Load the embedding model ----
# 'all-MiniLM-L6-v2' is a small but high-quality SBERT (Sentence-BERT) model.
# SBERT is BERT fine-tuned specifically to produce good SENTENCE embeddings.
# (Vanilla BERT was designed for token-level tasks; SBERT pools tokens into one vector.)
# On first run: downloads ~90MB from Hugging Face, then caches locally.
print(f"Loading embedding model '{EMBED_MODEL}'...")
print("(First run downloads ~90MB from Hugging Face — subsequent runs use cache)")
model = SentenceTransformer(EMBED_MODEL, device=DEVICE)
print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")
# WHY 384 dimensions? The model was designed to balance quality vs speed.
# Larger models (e.g., all-mpnet-base-v2) use 768 dims and are more accurate
# but encode ~3x slower. For teaching, 384 is ideal.

# ---- Encode all notes ----
# encode() returns a 2D array: shape (N_NOTES, 384)
# Each row is the embedding of one clinical note.
# batch_size=64: processes 64 notes at a time (uses GPU memory efficiently)
# show_progress_bar=True: shows a tqdm bar so you can watch progress
print(f"\nEncoding {N_NOTES} clinical notes into 384-dimensional vectors...")
print("(This is the GPU-accelerated step — should take ~5-15 seconds on RTX 4060)")
embeddings = model.encode(
    notes,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True  # WHY normalize? Cosine similarity on L2-normalized vectors
                                # equals dot product, which is what FAISS's IndexFlatIP uses.
                                # Normalization makes all vectors unit length (on the unit sphere),
                                # so distance = angle, nothing else.
)
print(f"Embeddings shape: {embeddings.shape}")
# Should print: (2000, 384)
# 2000 notes × 384 numbers per note

# ---- Quick sanity check: measure similarity between known-similar strings ----
print("\n--- Sanity check: semantic similarity between test phrases ---")
test_pairs = [
    ("The patient had a heart attack",   "Myocardial infarction documented"),
    ("The patient had a heart attack",   "Patient enjoys daily jogging"),
    ("Diabetes mellitus type 2",         "T2DM diagnosed"),
    ("Diabetes mellitus type 2",         "Patient has a broken leg"),
    ("Elevated blood pressure",          "Hypertension on problem list"),
]

for s1, s2 in test_pairs:
    v1 = model.encode([s1], normalize_embeddings=True)
    v2 = model.encode([s2], normalize_embeddings=True)
    sim = float(cosine_similarity(v1, v2)[0][0])
    verdict = "CLOSE (same meaning)" if sim > 0.65 else "FAR (different meaning)"
    print(f"  Similarity {sim:.3f} → {verdict}")
    print(f"    '{s1}'")
    print(f"    '{s2}'")

# ---- Build FAISS index ----
# IndexFlatIP = 'Inner Product' index.
# When vectors are L2-normalized (unit length), inner product = cosine similarity.
# 'Flat' = exact search (no approximation). Fast enough for 2,000 vectors.
# For millions of vectors, you'd use IndexIVFFlat or HNSW for approximate search.
DIM = embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(DIM)
index.add(embeddings.astype('float32'))  # FAISS requires float32
print(f"\nFAISS index built. Total vectors indexed: {index.ntotal}")
print("The index allows sub-millisecond nearest-neighbour lookup over all 2,000 notes.")

# ---- Define the search function ----
def semantic_search(query: str, k: int = N_RESULTS, show_output: bool = True) -> pd.DataFrame:
    """
    Find the k most semantically similar clinical notes to a free-text query.
    
    Steps:
    1. Encode the query string into a 384-dim vector
    2. Ask FAISS: 'which k stored vectors are most similar (highest cosine similarity)?'
    3. Return the matching notes with their similarity scores and patient metadata
    """
    # Step 1: Encode the query — same model, same normalization as the indexed notes
    # WHY same normalization? The query and index must live in the same space.
    query_vec = model.encode([query], normalize_embeddings=True).astype('float32')
    
    # Step 2: FAISS lookup — returns (scores, indices)
    # scores: cosine similarity (0 to 1 for normalized vectors)
    # indices: row positions in our sample_df
    scores, indices = index.search(query_vec, k)
    
    # Step 3: Assemble results DataFrame
    results = []
    for score, idx in zip(scores[0], indices[0]):
        patient = sample_df.iloc[idx]
        results.append({
            'similarity':    round(float(score), 4),
            'note_id':       int(idx),
            'heart_disease': int(patient['heart_disease']),
            'age':           int(patient['age']),
            'note_excerpt':  notes[idx][:150] + '...'
        })
    
    df_results = pd.DataFrame(results)
    
    if show_output:
        print(f"\n{'='*60}")
        print(f"QUERY: '{query}'")
        print(f"{'='*60}")
        for _, row in df_results.iterrows():
            hd_flag = "[CARDIAC HISTORY]" if row['heart_disease'] == 1 else ""
            print(f"\n  Similarity: {row['similarity']:.4f}  Age: {row['age']}  {hd_flag}")
            print(f"  {row['note_excerpt']}")
    
    return df_results

# ============================================================
# DEMONSTRATE SEMANTIC SEARCH
# ============================================================
# We run 4 queries to show what semantic search can and cannot do.
# Notice that queries use DIFFERENT words than the notes — that's the
# whole point. Keyword search would fail; embeddings succeed.

print("\n" + "="*60)
print("SEMANTIC SEARCH DEMONSTRATIONS")
print("These queries use DIFFERENT words than the stored notes.")
print("Keyword search would miss them. Embeddings find them.")

# Query 1: Uses 'coronary artery disease' — notes use 'heart attack', 'myocardial infarction', 'CAD'
q1_results = semantic_search(
    "older patient with coronary artery disease and high blood pressure",
    k=N_RESULTS
)

# Query 2: Uses 'diabetes' — notes use 'T2DM', 'diabetic', 'elevated glucose'
q2_results = semantic_search(
    "diabetic patient with glucose control issues",
    k=N_RESULTS
)

# Query 3: Low-risk patient description
q3_results = semantic_search(
    "young healthy non-smoker with normal vitals",
    k=N_RESULTS
)

# Query 4: Complex multi-condition — tests how well multi-concept queries work
q4_results = semantic_search(
    "elderly smoker with polypharmacy and cardiovascular history",
    k=N_RESULTS
)

print("\nAll 4 queries demonstrated. Proceeding to visualizations in Cell 5.")

In [ ]:
# ============================================================
# CELL 5: VISUALIZATIONS + SAVE OUTPUTS + SUMMARY
# ============================================================
# We produce 4 plots:
#  Plot 1: Embedding space — 2D PCA projection showing where notes cluster
#  Plot 2: Cosine similarity heatmap — shows the similarity matrix
#  Plot 3: Query results bar chart — similarity scores for the 4 queries
#  Plot 4: RAG concept diagram — how retrieval + generation work together

import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# ============================================================
# PLOT 1: Embedding Space (PCA 2D Projection)
# ============================================================
# The actual embedding space has 384 dimensions — impossible to visualize directly.
# PCA compresses it to 2D by finding the two directions of MOST variation.
# We lose information, but the CLUSTERS should still be visible:
# notes with similar content (e.g. diabetic patients) should cluster together.
#
# WHY PCA instead of t-SNE/UMAP here?
# PCA is LINEAR — it preserves global structure and is fast.
# t-SNE/UMAP are non-linear and better for tight cluster visualization,
# but they distort global distances. For this teaching plot, PCA is cleaner.

print("Generating Plot 1: Embedding space PCA projection...")
pca = PCA(n_components=2, random_state=SEED)
coords_2d = pca.fit_transform(embeddings)  # shape: (N_NOTES, 2)
explained = pca.explained_variance_ratio_

# Create a colour label based on heart disease status
colors_hd = [RED if hd == 1 else BLUE for hd in sample_df['heart_disease']]

fig1, axes1 = plt.subplots(1, 2, figsize=(16, 7))
fig1.suptitle(
    "Embedding Space: Clinical Notes Projected to 2D via PCA\n"
    "Each dot = one clinical note. Nearby dots = similar meaning.",
    fontsize=13, fontweight='bold'
)

# ---- Left panel: All notes coloured by heart disease ----
ax = axes1[0]
ax.scatter(
    coords_2d[:, 0], coords_2d[:, 1],
    c=colors_hd, alpha=0.35, s=12, linewidths=0
)
ax.set_xlabel(f"PCA Component 1 ({explained[0]*100:.1f}% variance)", fontsize=10)
ax.set_ylabel(f"PCA Component 2 ({explained[1]*100:.1f}% variance)", fontsize=10)
ax.set_title("All Notes — Coloured by Heart Disease Status", fontsize=11)

patch_no  = mpatches.Patch(color=BLUE, label=f"No heart disease (n={sum(sample_df['heart_disease']==0)})")
patch_yes = mpatches.Patch(color=RED,  label=f"Heart disease (n={sum(sample_df['heart_disease']==1)})")
ax.legend(handles=[patch_no, patch_yes], fontsize=9, loc='upper right')
ax.text(
    0.02, 0.02,
    f"Total explained variance: {(explained[0]+explained[1])*100:.1f}%\n"
    "(The remaining variance is spread\nacross 382 other dimensions)",
    transform=ax.transAxes, fontsize=8, color='grey',
    verticalalignment='bottom'
)

# ---- Right panel: Zoom in on a query + its nearest neighbours ----
# Show visually that the search results cluster near the query point
ax2 = axes1[1]
ax2.scatter(
    coords_2d[:, 0], coords_2d[:, 1],
    c=GREY, alpha=0.2, s=8, linewidths=0, zorder=1
)

# Plot the 5 nearest neighbours of Query 1 (cardiac query)
nn_indices = q1_results['note_id'].values
nn_coords  = coords_2d[nn_indices]
ax2.scatter(
    nn_coords[:, 0], nn_coords[:, 1],
    c=RED, s=80, zorder=3, label='Top-5 results:\ncardiac query', edgecolors='black', linewidths=0.5
)

# Encode the query itself and project to 2D using the same PCA
query_vec_raw = model.encode(["older patient with coronary artery disease and high blood pressure"],
                              normalize_embeddings=False)  # un-normalized for PCA
query_2d = pca.transform(query_vec_raw)
ax2.scatter(
    query_2d[0, 0], query_2d[0, 1],
    c=GREEN, s=200, marker='*', zorder=4, label='Query point', edgecolors='black', linewidths=0.8
)

# Draw lines from query to each result
for coord in nn_coords:
    ax2.plot(
        [query_2d[0, 0], coord[0]],
        [query_2d[0, 1], coord[1]],
        color=RED, alpha=0.4, linewidth=1, zorder=2
    )

ax2.set_xlabel(f"PCA Component 1 ({explained[0]*100:.1f}% variance)", fontsize=10)
ax2.set_ylabel(f"PCA Component 2 ({explained[1]*100:.1f}% variance)", fontsize=10)
ax2.set_title("Query + Nearest Neighbours in Embedding Space\n'\'cardiac + hypertension\' query → finds semantically close notes", fontsize=11)
ax2.legend(fontsize=9, loc='upper right')

plt.tight_layout()
p1_path = os.path.join(OUT_DIR, "17_01_embedding_space_pca.png")
plt.savefig(p1_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved: {p1_path}")

# ============================================================
# PLOT 2: Cosine Similarity Heatmap (Cross-Query Matrix)
# ============================================================
# Shows how similar our 4 search queries are to EACH OTHER.
# This teaches the geometry of embedding space:
# - Similar queries (cardiac + cardiac) → high similarity
# - Dissimilar queries (cardiac + healthy) → low similarity
#
# Also shows 6 test sentences from the sanity check above — the
# matrix should confirm that synonym pairs cluster together.

print("\nGenerating Plot 2: Cosine similarity heatmap...")

demo_sentences = [
    "Coronary artery disease query",
    "Diabetic patient query",
    "Healthy young patient",
    "Elderly smoker cardiac",
    "heart attack",
    "myocardial infarction",
    "diabetes mellitus",
    "T2DM elevated glucose",
    "healthy non-smoker",
    "young athlete fitness",
]

demo_vecs = model.encode(demo_sentences, normalize_embeddings=True)
sim_matrix = cosine_similarity(demo_vecs)  # shape: (10, 10)

fig2, ax3 = plt.subplots(figsize=(11, 9))
im = ax3.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax3, label='Cosine Similarity (0 = unrelated, 1 = identical meaning)')

ax3.set_xticks(range(len(demo_sentences)))
ax3.set_yticks(range(len(demo_sentences)))
short_labels = [
    "CAD query", "Diabetes query", "Healthy query", "Elderly smoker",
    "heart attack", "myocardial infarction",
    "diabetes mellitus", "T2DM glucose",
    "healthy non-smoker", "young athlete"
]
ax3.set_xticklabels(short_labels, rotation=40, ha='right', fontsize=9)
ax3.set_yticklabels(short_labels, fontsize=9)

# Annotate cells with similarity values
for i in range(len(demo_sentences)):
    for j in range(len(demo_sentences)):
        val = sim_matrix[i, j]
        text_color = 'black' if 0.3 < val < 0.85 else 'white'
        ax3.text(j, i, f"{val:.2f}", ha='center', va='center', fontsize=8, color=text_color)

ax3.set_title(
    "Cosine Similarity Heatmap — How Similar Are These Phrases?\n"
    "Green = semantically close (same topic). Red/yellow = distant (different topic).\n"
    "Notice: 'heart attack' and 'myocardial infarction' score ~0.85+",
    fontsize=12, fontweight='bold'
)

# Highlight the synonym block
for box_coords, label in [
    ((3.5, 3.5, 2, 2), "Cardiac synonyms\nshould be similar"),
    ((5.5, 5.5, 2, 2), "Diabetes synonyms\nshould be similar"),
]:
    x0, y0, w, h = box_coords
    rect = plt.Rectangle((x0, y0), w, h, linewidth=2, edgecolor='navy', facecolor='none', linestyle='--')
    ax3.add_patch(rect)

plt.tight_layout()
p2_path = os.path.join(OUT_DIR, "17_02_cosine_similarity_heatmap.png")
plt.savefig(p2_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved: {p2_path}")

# ============================================================
# PLOT 3: Query Result Similarity Scores (Bar Charts)
# ============================================================
# For each of the 4 queries, show the top-5 similarity scores.
# Teach what a 'good' score looks like (>0.75) vs a 'marginal' one.
# Also show whether results are cardiac cases — does the search find
# heart disease notes when we ask about cardiac topics?

print("\nGenerating Plot 3: Query result similarity scores...")

query_labels = [
    "CAD + hypertension",
    "Diabetic glucose control",
    "Young healthy non-smoker",
    "Elderly smoker polypharmacy"
]
all_results = [q1_results, q2_results, q3_results, q4_results]

fig3, axes3 = plt.subplots(2, 2, figsize=(15, 10))
fig3.suptitle(
    "Top-5 Semantic Search Results per Query\n"
    "Bar height = cosine similarity. Red bars = patients with heart disease history.",
    fontsize=13, fontweight='bold'
)

for ax, res_df, qlabel in zip(axes3.flat, all_results, query_labels):
    bar_colors = [RED if hd == 1 else BLUE for hd in res_df['heart_disease']]
    bars = ax.bar(
        range(len(res_df)),
        res_df['similarity'],
        color=bar_colors, edgecolor='black', linewidth=0.5
    )
    # Annotate similarity score and age above each bar
    for i, (_, row) in enumerate(res_df.iterrows()):
        ax.text(
            i, row['similarity'] + 0.005,
            f"{row['similarity']:.3f}\nAge {row['age']}",
            ha='center', va='bottom', fontsize=8
        )
    ax.set_ylim(0, 1.1)
    ax.axhline(0.75, color='green', linestyle='--', alpha=0.5, linewidth=1)
    ax.axhline(0.5,  color='orange', linestyle='--', alpha=0.5, linewidth=1)
    ax.text(4.6, 0.76, '>0.75\nStrong', fontsize=7, color='green', ha='right')
    ax.text(4.6, 0.51, '>0.5\nModerate', fontsize=7, color='orange', ha='right')
    ax.set_xticks(range(len(res_df)))
    ax.set_xticklabels([f"Result {i+1}" for i in range(len(res_df))], fontsize=9)
    ax.set_ylabel("Cosine Similarity", fontsize=9)
    ax.set_title(f"Query: '{qlabel}'", fontsize=10, fontweight='bold')

    patch_hd  = mpatches.Patch(color=RED,  label="Has heart disease")
    patch_no  = mpatches.Patch(color=BLUE, label="No heart disease")
    ax.legend(handles=[patch_hd, patch_no], fontsize=8, loc='lower right')

plt.tight_layout()
p3_path = os.path.join(OUT_DIR, "17_03_query_results_similarity.png")
plt.savefig(p3_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved: {p3_path}")

# ============================================================
# PLOT 4: RAG Architecture Diagram
# ============================================================
# RAG = Retrieval-Augmented Generation.
# This is how AI assistants like Claude or ChatGPT work when you give
# them access to a document library (e.g. a hospital's clinical guidelines).
#
# PROBLEM: An LLM (language model) was trained up to a cutoff date.
#          It doesn't know YOUR documents, YOUR patients, YOUR data.
# SOLUTION: Before the LLM generates an answer:
#   1. Embed the user's question
#   2. Search the document database for the most relevant chunks
#   3. Put those chunks INTO the prompt as context
#   4. The LLM now has relevant, up-to-date information to draw on
#
# This script has built the RETRIEVAL part (steps 1-3) of a RAG pipeline.
# Script 18 (Generative AI) adds the GENERATION part (step 4).

print("\nGenerating Plot 4: RAG architecture concept diagram...")

fig4, ax4 = plt.subplots(figsize=(14, 8))
ax4.set_xlim(0, 10)
ax4.set_ylim(0, 7)
ax4.axis('off')
ax4.set_facecolor('#F8F9FA')
fig4.patch.set_facecolor('#F8F9FA')

ax4.set_title(
    "RAG: Retrieval-Augmented Generation — How AI Systems Use Semantic Search\n"
    "This script builds the 'Retrieval' component. Script 18 adds the 'Generation' component.",
    fontsize=13, fontweight='bold', pad=12
)

# ---- Helper to draw a rounded box ----
def draw_box(ax, x, y, w, h, text, color, fontsize=10, text_color='white'):
    box = mpatches.FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle="round,pad=0.1",
        facecolor=color, edgecolor='white', linewidth=1.5
    )
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            color=text_color, fontweight='bold', wrap=True,
            multialignment='center')

# ---- Arrow helper ----
def draw_arrow(ax, x1, y1, x2, y2, label='', color='#555'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx+0.05, my+0.15, label, fontsize=8, color=color, style='italic')

# ---- OFFLINE INDEXING PIPELINE (top row) ----
ax4.text(5, 6.6, '① OFFLINE: Build the Index (done once)', ha='center', fontsize=11,
         color='#1565C0', fontweight='bold')

draw_box(ax4, 1.2, 5.8, 1.8, 0.8, "Clinical\nDocuments", '#607D8B', fontsize=9)
draw_box(ax4, 3.5, 5.8, 2.0, 0.8, "Embedding\nModel\n(SBERT)", '#1565C0', fontsize=9)
draw_box(ax4, 6.0, 5.8, 2.0, 0.8, "384-dim\nVectors", '#1565C0', fontsize=9)
draw_box(ax4, 8.5, 5.8, 1.8, 0.8, "FAISS\nVector Index", '#0D47A1', fontsize=9)

draw_arrow(ax4, 2.15, 5.8, 2.5, 5.8, 'chunk text')
draw_arrow(ax4, 4.5,  5.8, 5.0, 5.8, 'encode')
draw_arrow(ax4, 7.0,  5.8, 7.6, 5.8, 'store')

# ---- ONLINE QUERY PIPELINE (bottom rows) ----
ax4.text(5, 4.7, '② ONLINE: Answer a Query (happens per request)', ha='center', fontsize=11,
         color='#B71C1C', fontweight='bold')

# User query
draw_box(ax4, 1.2, 3.9, 1.8, 0.8, "User\nQuestion", '#E53935', fontsize=9)
draw_box(ax4, 3.5, 3.9, 2.0, 0.8, "Same\nEmbedding\nModel", '#C62828', fontsize=9)
draw_box(ax4, 6.0, 3.9, 2.0, 0.8, "Query\nVector", '#C62828', fontsize=9)
draw_box(ax4, 8.5, 3.9, 1.8, 0.8, "FAISS\nSearch", '#B71C1C', fontsize=9)

draw_arrow(ax4, 2.15, 3.9, 2.5, 3.9, 'encode')
draw_arrow(ax4, 4.5,  3.9, 5.0, 3.9, '')
draw_arrow(ax4, 7.0,  3.9, 7.6, 3.9, 'lookup')

# Top-k results flow down
draw_box(ax4, 8.5, 2.8, 1.8, 0.8, "Top-K\nRelevant\nNotes", '#FF6F00', fontsize=9)
draw_arrow(ax4, 8.5, 3.5, 8.5, 3.15, 'k=5 results')

# Combine into LLM prompt
draw_box(ax4, 5.5, 2.0, 2.4, 0.9,
         "Prompt = Question\n+ Retrieved Context", '#E65100', fontsize=9)
draw_arrow(ax4, 7.6, 2.8, 6.7, 2.4, 'inject context')
draw_arrow(ax4, 1.2, 3.5, 4.4, 2.4, 'add question')

# LLM generates answer
draw_box(ax4, 5.5, 0.9, 2.4, 0.8, "LLM\n(GPT / Claude)", '#1B5E20', fontsize=9)
draw_box(ax4, 5.5, 0.05, 2.4, 0.6, "Final Answer", '#2E7D32', fontsize=8)
draw_arrow(ax4, 5.5, 1.55, 5.5, 1.3, 'generate')
draw_arrow(ax4, 5.5, 0.5, 5.5, 0.36, '')

# ---- Annotation ----
ax4.text(
    0.5, 0.2,
    "This script builds the blue/orange components.\nScript 18 adds the green LLM component.",
    fontsize=9, color='#555', style='italic'
)

plt.tight_layout()
p4_path = os.path.join(OUT_DIR, "17_04_rag_architecture_diagram.png")
plt.savefig(p4_path, dpi=150, bbox_inches='tight', facecolor='#F8F9FA')
plt.show()
print(f"Saved: {p4_path}")

# ============================================================
# SUMMARY + WHAT WE LEARNED
# ============================================================

print("\n" + "="*60)
print("SCRIPT 17 COMPLETE — Embeddings and Semantic Search")
print("="*60)

print(f"""
WHAT WE BUILT:
  • Generated {N_NOTES} synthetic clinical notes from {N_NOTES} patients in our dataset
  • Encoded all notes into 384-dimensional embedding vectors using SBERT
  • Built a FAISS vector index for sub-millisecond nearest-neighbour search
  • Demonstrated 4 semantic queries — each using DIFFERENT words than the notes
  • Visualised the embedding space, similarity matrix, and query results
  • Explained the RAG architecture that connects search to generation

KEY TAKEAWAYS:
  1. An embedding is a point in high-dimensional space where meaning determines location.
     'Myocardial infarction' and 'heart attack' end up NEAR each other because the model
     was trained on text where they appeared in identical contexts.

  2. Cosine similarity measures the ANGLE between two vectors, not distance.
     This makes it robust to text length — a short summary and a long note
     about the same topic score as similar even though one is much longer.

  3. Keyword search fails with synonym-rich clinical text.
     A doctor searching for 'heart failure' misses 'CHF', 'congestive failure',
     'cardiomyopathy', 'reduced EF'. Embedding search finds all of them.

  4. FAISS makes vector search fast at scale.
     Brute-force comparison against 1M vectors takes seconds.
     FAISS's approximate indexes (HNSW, IVFFlat) handle 100M+ vectors in milliseconds.

  5. This script builds the RETRIEVAL component of RAG.
     RAG = find the most relevant documents (what we just did) + give them to an LLM
     as context (Script 18). This is how AI assistants answer questions about
     YOUR data, YOUR documents — without retraining.

  6. Embeddings generalise beyond text.
     The same concept applies to images (image search), audio, molecules,
     and even tabular records. Any domain where 'similarity' matters can
     be represented as vectors in a learned embedding space.

HOW THIS CONNECTS TO THE R SCRIPTS:
  In scripts 01-15 we worked with STRUCTURED data — rows and columns.
  Embedding models turn UNSTRUCTURED data (text) into structured vectors.
  Once text is a vector, all the distance metrics, clustering algorithms,
  and classifiers from the R scripts apply directly.

WHAT KINDS OF PROBLEMS THIS TECHNIQUE ADDRESSES:
  • Clinical NLP: finding relevant prior cases, similar diagnoses, drug interactions
  • Document search: finding relevant pages in a 10,000-page policy library
  • Deduplication: finding near-duplicate patient records across systems
  • Question answering over private data (the RAG pattern)
  • Recommendation: 'patients like this one' for treatment matching
  • Anomaly detection: notes that are far from all others may be data entry errors

NEXT: Script 18 (Generative AI) adds the LLM generation step to complete the RAG loop.
""")

print("Plots saved to:", OUT_DIR)
print("  17_01_embedding_space_pca.png")
print("  17_02_cosine_similarity_heatmap.png")
print("  17_03_query_results_similarity.png")
print("  17_04_rag_architecture_diagram.png")